In [1]:
# -*- coding: utf-8 -*-


# --- BEGIN TAGS PATCH (must run before sklearn is imported) ---
# --- BEGIN TAGS PATCH (must run before any SearchCV) ---
from sklearn.base import BaseEstimator
if not hasattr(BaseEstimator, "__sklearn_tags__"):
    BaseEstimator.__sklearn_tags__ = lambda self: {}

from scikeras.wrappers import KerasRegressor
if not hasattr(KerasRegressor, "__sklearn_tags__"):
    # 返回一个空 dict 或者根据需要设置标签
    KerasRegressor.__sklearn_tags__ = lambda self: {}
# --- END TAGS PATCH ---

import joblib
import os
# 一定要在 import tensorflow 之前设置
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0=ALL, 1=INFO, 2=WARNING, 3=ERROR
import os
# from openpyxl import load_workbook
from openpyxl import Workbook, load_workbook

EXCEL_PATH = 'model_results.xlsx'

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    mean_squared_log_error,
    explained_variance_score,
    max_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from skopt import BayesSearchCV
from tensorflow.keras import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense
from tensorflow.keras.layers import GRU
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
# from tensorflow.keras.wrappers.scikit_learn import KerasRegressor


from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from skopt import BayesSearchCV
import time
import argparse
from skopt.space import Real, Integer, Categorical
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 只保留 ERROR，不显示 INFO、WARNING

# 然后再 import
import warnings
warnings.filterwarnings('ignore')  # 全局忽略所有 warnings

import logging
tf.get_logger().setLevel(logging.ERROR)     # Keras/TF 自己的 logger
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# parser = argparse.ArgumentParser()
# parser.add_argument('--seed', type=int, default=42,
#                     help='random seed')
# args = parser.parse_args()
# random_seed = args.seed

from sklearn.model_selection import GridSearchCV


print('-------------start!---------------------')

# === 1. read data
xls = pd.read_excel(
    '../ParleysRWIS_20211101-20230430.xlsx',
    sheet_name=['RwisAtmosphericData', 'RwisSurfaceData'],
    parse_dates=['SampleTime'],
    engine='openpyxl'
)
df1 = xls['RwisAtmosphericData']
df2 = xls['RwisSurfaceData']
df1['SampleTime'] = pd.to_datetime(df1['SampleTime'])
df2['SampleTime'] = pd.to_datetime(df2['SampleTime'])
df1['RwisControllerIntId'] = df1['RwisControllerIntId'].astype(str)
df2['RwisControllerIntId'] = df2['RwisControllerIntId'].astype(str)
df = pd.merge(df1, df2, on=['SampleTime','RwisControllerIntId'], how='inner', suffixes=('_atm','_surf'))

# === 2. delete useless feature
cols_to_drop = [
    'Pressure', 'SnowIntervalDepth','PrecipitationIntervalAmount','PrecipitationAmount',
    'RoadStatusText','iceCurCondIndex','iceAvgCondIndex','iceCurCondCode','iceAvgCondCode',
    'iceCurFricIndex','iceAvgFricIndex','iceCurFricCode','iceAvgFricCode','iceGrip','iceXValue',
    'iceYValue','iceYXRatio','iceLens','SurfaceSensorError','SurfaceSalinity','SurfaceFreezePoint','RoadStatus',
    'RoadStatus','RoadTemp','iceRoadTemp','RwisAtmoEquipmentIntId','WindSpeedGustTimestamp',
    'RwisSurfaceEquipmentIntId','WindDirection','WindSpeedAverage','WindSpeedGust','Visibility',
    'BatteryVoltage','dstInputVoltage','dstHardwareStatus','SurfaceStatusText','dscRoadStatusText','SubSurfaceTemp',
    'AirTemp_atm','DewPoint_atm','RelativeHumidity_atm','dscLevelOfGrip','dstRoadTemp','WinterRoadWeatherIndex',
    'StormIntensityIndex','PerformanceMetric','dscSurfStatus','WetBulbTemp'
]
df = df.drop(columns=cols_to_drop)

# === 3. remove outliers
filters = [
    (df['PrecipitationIntensity'] != 'fog'),
#     (df['WetBulbTemp'] >= -100),
    (df['SurfaceTemp'] < 200),
    (df['AirTemp_surf'] < 100),
    (df['DewPoint_surf'] < 80) & (df['DewPoint_surf'] > -20),
#     (df['dscSurfStatus'] < 10),
    (df['RainTotal'] <= 0.1),
    (df['SnowfallRate'] <= 3),
#     (df['WinterRoadWeatherIndex'] <= 1.6),
#     (df['StormIntensityIndex'] <= 2.2),
    (df['SurfaceWaterDepth'] <= 2),
    (df['SurfaceIceDepth'] <= 0.4),
    (df['SurfaceSnowDepth'] <= 2)
]
for f in filters:
    df = df[f]
df = df.fillna(df.mean(numeric_only=True))
df = df[(df['SurfaceGrip'] >= 0) & (df['SurfaceGrip'] <= 1)].reset_index(drop=True)

# === 4. choose time diff=10min data
df = df.sort_values(['RwisControllerIntId','SampleTime']).reset_index(drop=True)
df['time_diff'] = df.groupby('RwisControllerIntId')['SampleTime'].diff()
mask5 = ((df['time_diff']==pd.Timedelta(minutes=5)) & (df['SampleTime'].dt.minute%10==5))
df2 = df[~mask5].reset_index(drop=True)
df2 = df2.sort_values(['RwisControllerIntId','SampleTime']).reset_index(drop=True)
df2['time_diff'] = df2.groupby('RwisControllerIntId')['SampleTime'].diff()
df10 = df2[df2['time_diff']==pd.Timedelta(minutes=10)].reset_index(drop=True)

# === 5. One-hot, minmax
cat_cols = ['PrecipitationIntensity','PrecipitationType']
df_enc = pd.get_dummies(df10, columns=cat_cols)
label_col = 'SurfaceGrip'
feature_cols = [c for c in df_enc.columns if c not in ['SampleTime','RwisControllerIntId','time_diff',label_col]]
scaler = MinMaxScaler()
df_enc[feature_cols] = scaler.fit_transform(df_enc[feature_cols])

print('-------------finish data preprocessing!---------------------')
# === 5b. save preprocessed data so you can skip steps 1–5 next time ===
PREPROCESSED_PATH = '../preprocessed_rwis_data.xlsx'
df_enc.to_excel(PREPROCESSED_PATH, index=False)
print(f"Preprocessed data saved to {PREPROCESSED_PATH}")



-------------start!---------------------
-------------finish data preprocessing!---------------------
Preprocessed data saved to ../preprocessed_rwis_data.xlsx


In [3]:
# -*- coding: utf-8 -*-
import os
import time

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import (
    train_test_split,
    LeaveOneGroupOut,
    GridSearchCV
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, median_absolute_error,
    mean_squared_log_error, explained_variance_score,
    max_error, r2_score
)
from scikeras.wrappers import KerasRegressor
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

import joblib
from openpyxl import Workbook, load_workbook

# === 全局常量 ===
PREPROCESSED_PATH = '../preprocessed_rwis_data.xlsx'
WINDOW = 4        # 过去多少步用来预测
HORIZON = 1       # 预测步长（如果是一步预测，就用 1）
SEEDS = (42, 43, 44)

# === 1. 载入预处理好的 DataFrame ===
df_enc = pd.read_excel(PREPROCESSED_PATH, engine='openpyxl')
label_col   = 'SurfaceGrip'
feature_cols = [
    c for c in df_enc.columns
    if c not in ['SampleTime','RwisControllerIntId','time_diff', label_col]
]

# === 2. 构建 dataset 的函数 ===
def build_rnn_dataset(df, feature_cols, label_col='SurfaceGrip', window=WINDOW):
    X_seq, y_seq, groups = [], [], []
    for gid, grp in df.groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            hist = grp.iloc[i-window:i]
            tgt  = grp.iloc[i]
            feat = hist[feature_cols].values
            hy   = hist[[label_col]].values
            seq_hist  = np.hstack([feat, hy])
            curr_feat = tgt[feature_cols].values.reshape(1,-1)
            curr_all  = np.hstack([curr_feat, hy[-1:]])
            seq       = np.vstack([seq_hist, curr_all])
            X_seq.append(seq)
            y_seq.append(tgt[label_col])
            groups.append(gid)
    return np.array(X_seq), np.array(y_seq), np.array(groups)

def build_tabular_dataset(df, feature_cols, label_col='SurfaceGrip', window=WINDOW):
    X_tab, y_tab, groups = [], [], []
    for gid, grp in df.sort_values(['RwisControllerIntId','SampleTime']).groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            tgt = grp.iloc[i]
            X_tab.append(tgt[feature_cols].values)
            y_tab.append(tgt[label_col])
            groups.append(gid)
    return np.array(X_tab), np.array(y_tab), np.array(groups)

# === 3. balance helper ===
def balance_y_distribution(X_seq, X_tab, y, bins=50, max_per_bin=1000, random_state=42):
    np.random.seed(random_state)
    edges = np.linspace(np.min(y), np.max(y), bins+1)
    idx   = np.digitize(y, bins=edges, right=True)
    sel   = []
    for b in range(1, bins+1):
        ids = np.where(idx==b)[0]
        if len(ids) > max_per_bin:
            ids = np.random.choice(ids, max_per_bin, replace=False)
        sel.extend(ids)
    sel = np.array(sel)
    return X_seq[sel], X_tab[sel], y[sel]

# === 4. 评价指标 ===
def evaluate(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    mae   = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true, y_pred)
    y_pred_pos = np.maximum(y_pred, 0)
    msle  = mean_squared_log_error(y_true, y_pred_pos)
    rmsle = np.sqrt(msle)
    smape = np.mean(
        np.abs(y_true - y_pred) /
        ((np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8)
    )
    if len(y_true) > 1:
        naive_mae = np.mean(np.abs(np.diff(y_true)))
        mase = mae / (naive_mae + 1e-8)
    else:
        mase = np.nan
    ev  = explained_variance_score(y_true, y_pred)
    mx  = max_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    return {
        'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MedAE': medae,
        'MAPE': mape, 'sMAPE': smape, 'MSLE': msle, 'RMSLE': rmsle,
        'MASE': mase, 'ExplVar': ev, 'MaxError': mx, 'R2': r2
    }

# === 5. Excel helper ===
def init_model_excel(model_name):
    path = f'{model_name}_results.xlsx'
    if not os.path.exists(path):
        wb = Workbook()
        ws = wb.active
        ws.title = 'Metrics'
        headers = list(evaluate([0],[0]).keys()) + ['fold','station','dataset']
        ws.append(headers)
        wb.save(path)
    return path

def append_model_metrics(model_name, row_dict):
    path = init_model_excel(model_name)
    wb = load_workbook(path)
    ws = wb['Metrics']
    header = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    row = [row_dict.get(col, "") for col in header]
    ws.append(row)
    wb.save(path)

# === 6. LSTM 训练函数 ===
def train_lstm(X, y):
    def build_model(lr=1e-3):
        m = Sequential([
            LSTM(64, input_shape=X.shape[1:]),
            Dense(HORIZON)
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse')
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)]
    )
    param_grid = {'epochs': [200,300,400]}
    grid = GridSearchCV(
        keras_reg, param_grid,
        scoring='neg_mean_squared_error',
        cv=3, n_jobs=1
    )
    grid.fit(X, y)
    return grid.best_estimator_, grid.best_estimator_.history_

# === 7. LOOCV 函数 ===
def loocv_run(X_seq, X_tab, y, groups, tab_fns, seq_fns, seed):
    logo = LeaveOneGroupOut()
    for fold, (tr, te) in enumerate(logo.split(X_tab, y, groups), start=1):
        station = groups[te][0]
        Xs_tr, Xt_tr, y_tr = X_seq[tr], X_tab[tr], y[tr]
        Xs_te, Xt_te, y_te = X_seq[te], X_tab[te], y[te]

        # 重新平衡
        Xs_tr, Xt_tr, y_tr = balance_y_distribution(Xs_tr, Xt_tr, y_tr, random_state=seed)

        # 划分验证集
        Xs_tr_sub, Xs_val, y_tr_sub_s, y_val_s = train_test_split(
            Xs_tr, y_tr, test_size=0.2, random_state=seed
        )
        Xt_tr_sub, Xt_val, y_tr_sub_t, y_val_t = train_test_split(
            Xt_tr, y_tr, test_size=0.2, random_state=seed
        )

        # Tabular
        for name, fn in tab_fns:
            mdl = fn(Xt_tr_sub, y_tr_sub_t)
            for ds, Xv, yv in [('val', Xt_val, y_val_t), ('test', Xt_te, y_te)]:
                y_pred = mdl.predict(Xv)
                mets = evaluate(yv, y_pred)
                mets.update(model=name, fold=fold, station=station, dataset=ds)
                append_model_metrics(name, mets)
            os.makedirs(f'models/{name}', exist_ok=True)
            joblib.dump(mdl, f'models/{name}/{name}_fold{fold}.joblib', compress=3)

        # Sequence
        for name, fn in seq_fns:
            mdl, hist = fn(Xs_tr_sub, y_tr_sub_s)
            for ds, Xv, yv in [('val', Xs_val, y_val_s), ('test', Xs_te, y_te)]:
                y_pred = mdl.predict(Xv).ravel()
                mets = evaluate(yv, y_pred)
                mets.update(model=name, fold=fold, station=station, dataset=ds)
                append_model_metrics(name, mets)
            # 保存 training history
            df_hist = pd.DataFrame(hist)
            with pd.ExcelWriter(f'{name}_results.xlsx', engine='openpyxl',
                                 mode='a', if_sheet_exists='replace') as writer:
                df_hist.to_excel(writer, sheet_name=f'History_Fold{fold}',
                                 index_label='epoch')
            os.makedirs(f'models/{name}', exist_ok=True)
            mdl.model_.save(f'models/{name}/{name}_fold{fold}.h5')

# === 8. 主循环：三种 seed ===
# 先构建好一次 dataset
X_seq, y_seq, groups_seq = build_rnn_dataset(df_enc, feature_cols, label_col)
X_tab, y_tab, groups_tab = build_tabular_dataset(df_enc, feature_cols, label_col)
assert np.array_equal(groups_seq, groups_tab)

for seed in SEEDS:
    print(f"\n\n===== RUNNING SEED {seed} =====")
    np.random.seed(seed)
    tf.random.set_seed(seed)

    tab_fns = [
        # ('RF_seed'+str(seed), train_rf), …
    ]
    seq_fns = [
        (f'LSTM_seed_length4_{seed}', train_lstm),
    ]

    tf.autograph.set_verbosity(0)
    os.makedirs('models', exist_ok=True)
    start = time.time()
    loocv_run(X_seq, X_tab, y_tab, groups_seq, tab_fns, seq_fns, seed)
    print(f"Seed {seed} done in {time.time() - start:.1f}s")



===== RUNNING SEED 42 =====
Seed 42 done in 11880.9s


===== RUNNING SEED 43 =====
Seed 43 done in 9696.8s


===== RUNNING SEED 44 =====
Seed 44 done in 8792.6s
